# SAM 3: Köpfe in Patientenvideos finden und verpixeln

Test, ob SAM 3 (Text-Prompt + Segmentierung) Köpfe zuverlässiger findet als SCRFD/YOLO11.

**Ablauf nach einem Kernel-Neustart:** Abschnitte **0**, **1** und **1b** von oben nach unten ausführen, dann in **2** den gewünschten Lauf.
- **1** = SAM 3 mit eigenem Tracking (`run_video_split`)
- **1b** = SAM 3 pro Frame ohne Tracking (`run_video_detect`)

Den **Anhang** nicht im normalen Ablauf ausführen (lädt zum Teil weitere Modelle).

**Vor jedem Commit: alle Ausgaben löschen** (Patientendaten!).

## 0. Setup

In [ ]:
from pathlib import Path
import pickle
import time

import cv2
import numpy as np
import torch
from transformers import Sam3VideoModel, Sam3VideoProcessor

PROJECT_ROOT = Path.cwd().parent          # Notebook liegt in notebooks/, also eins höher
MODEL_DIR    = PROJECT_ROOT / "models" / "sam3"
INPUT_VIDEO  = PROJECT_ROOT / "data" / "raw" / "PAT_1" / "VID_76.AVI"
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim" / "PAT_1"     # Zwischenergebnisse (.pkl)
OUTPUT_DIR   = PROJECT_ROOT / "data" / "processed" / "PAT_1"   # fertige Videos/Bilder
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda"

for p in [MODEL_DIR, INPUT_VIDEO]:
    assert p.exists(), f"Fehlt: {p}"

print("torch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Video-Modell laden (ca. 1,8 GB GPU-Speicher)
video_processor = Sam3VideoProcessor.from_pretrained(MODEL_DIR)
video_model = Sam3VideoModel.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to(DEVICE).eval()
print(f"GPU-Speicher belegt: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### Speicher aufräumen

Die Sitzung hebt sonst die Zwischenergebnisse **jedes** Frames auf (~10–13 MB GPU pro Frame). Ab ~6,5 GB lagert Windows aus, ein Frame dauert dann ~12 s statt ~0,6 s.

Der Tracker liest aber nur wenige zurückliegende Frames (siehe `transformers/models/sam3_tracker_video/modeling_sam3_tracker_video.py`):

| Was | wie weit zurück | Einstellung in `models/sam3/config.json` |
|---|---|---|
| Speicher-Features | 6 Frames | `num_maskmem = 7` |
| Objekt-Zeiger | 15 Frames | `max_object_pointers_in_encoder = 16` |
| Startphase neuer Objekte | 15 Frames | `hotstart_delay = 15` |
| Frames mit Neuerkennung | die 4 nächsten | `max_cond_frame_num = 4` |

Alles, was älter als `KEEP = 30` Frames ist, wird daher gelöscht.

In [ ]:
KEEP = 30   # so viele zurückliegende Frames behalten (Tracker braucht max. ~16)

def prune_session(session, current_idx, keep=KEEP):
    """Entfernt Zwischenergebnisse und Frames, die der Tracker nicht mehr braucht."""
    cutoff = current_idx - keep

    # 1. Vorverarbeitete Frames (RAM)
    for idx in [k for k in session.processed_frames if k < cutoff]:
        del session.processed_frames[idx]

    for obj_out in session.output_dict_per_obj.values():
        # 2. Normale Tracking-Ergebnisse (GPU): alles älter als cutoff
        non_cond = obj_out["non_cond_frame_outputs"]
        for idx in [k for k in non_cond if k < cutoff]:
            del non_cond[idx]

        # 3. Frames mit Neuerkennung: nur die 4 neuesten behalten
        cond = obj_out["cond_frame_outputs"]
        for idx in sorted(cond)[:-4]:
            if idx < cutoff:
                del cond[idx]

### Ersatz für das `kernels`-Modul (NMS, Löcher füllen, Sprenkel entfernen)

SAM 3 lädt für drei Nachbearbeitungsschritte ein fertig kompiliertes Modul (`kernels-community/cv-utils`). Das gibt es **nur für Linux**. Unter Windows überspringt SAM 3 die Schritte stillschweigend (siehe `transformers/models/sam3_video/modeling_sam3_video.py`, `nms_masks` und `fill_holes_in_mask_scores`):

| Schritt | Wofür | ohne Modul |
|---|---|---|
| **NMS** | doppelte Erkennungen desselben Objekts zusammenfassen | alle Erkennungen werden behalten → Doppelgänger-IDs |
| **Löcher füllen** | kleine Löcher in einer Maske schließen | übersprungen |
| **Sprenkel entfernen** | kleine abgetrennte Maskenstücke löschen | übersprungen |

Der Ersatz unten macht dasselbe mit PyTorch (NMS) und OpenCV (`connectedComponentsWithStats` für zusammenhängende Flächen) und wird SAM 3 an der Stelle des Moduls untergeschoben. Getestet gegen die SAM-3-Funktionen `nms_masks` und `fill_holes_in_mask_scores`.

In [ ]:
import transformers.models.sam3_video.modeling_sam3_video as sam3_video_module


class CvUtilsErsatz:
    """Ersatz für 'kernels-community/cv-utils' (nur Linux): gleiche zwei Funktionen, gleiche Ein-/Ausgaben."""

    @staticmethod
    def generic_nms(ious, probs, iou_threshold, use_iou_matrix=True):
        """Greedy-NMS: Erkennung mit höchstem Score behalten, alle stark überlappenden verwerfen, wiederholen.
        Gibt die Indizes der behaltenen Erkennungen zurück."""
        order = torch.argsort(probs.float(), descending=True).cpu().tolist()
        ious_cpu = ious.float().cpu()
        suppressed = torch.zeros(len(order), dtype=torch.bool)
        keep = []
        for idx in order:
            if suppressed[idx]:
                continue
            keep.append(idx)
            suppressed |= ious_cpu[idx] > iou_threshold
        return torch.tensor(keep, dtype=torch.long, device=probs.device)

    @staticmethod
    def cc_2d(mask, get_counts=True):
        """Zusammenhängende Flächen in Masken der Form (N, 1, H, W).
        labels: Nummer der Fläche je Pixel, counts: Größe der Fläche, zu der das Pixel gehört."""
        m = mask.detach().to("cpu").numpy().astype(np.uint8)
        labels = np.zeros(m.shape, np.int32)
        counts = np.zeros(m.shape, np.int32)
        for n in range(m.shape[0]):
            for c in range(m.shape[1]):
                _, lab, stats, _ = cv2.connectedComponentsWithStats(m[n, c], connectivity=8)
                area = stats[:, cv2.CC_STAT_AREA].astype(np.int32)
                cnt = area[lab]
                cnt[m[n, c] == 0] = 0
                labels[n, c] = lab
                counts[n, c] = cnt
        return torch.from_numpy(labels).to(mask.device), torch.from_numpy(counts).to(mask.device)


# SAM 3 benutzt ab jetzt den Ersatz (statt "kernels nicht installiert -> überspringen")
sam3_video_module.cv_utils_kernel = CvUtilsErsatz()
print("NMS-Ersatz aktiv, IoU-Schwelle für Duplikate:", video_model.config.det_nms_thresh)

## 1. Getrennte Ansichten: Farbe und Schwarz-Weiß

Das Video enthält **zwei Kameras**: Die Farbansicht füllt das ganze Bild, die S/W-Ansicht ist als Fenster **oben rechts** eingeblendet (etwa ein Viertel des Bildes, links mit schmalem schwarzem Balken). Gemessen über Farbsättigung und Helligkeit von 60 Frames: S/W-Bild x 981–1919, y 0–527, schwarzer Balken x 961–980.

Auf dem ganzen Bild hielt SAM 3 das Fenster für einen Teil des Raums: Masken liefen über die Fensterkante, zeitweise wurden bis zu 11 Objekte verfolgt, und ein Frame dauerte bis zu 10 s.

**Deshalb** bekommt jede Ansicht eine **eigene SAM-3-Sitzung**:

```
x=0                  961  991                1910
 ┌────────────────────┐    ┌──────────────────┐   y=10
 │                    │    │  SW (919 × 517)  │
 │  F: Farbe          │    └──────────────────┘   y=527
 │  (961 × 1080)      │       ███ schwarz ███
 └────────────────────┘                           y=1079
```

- **F (Farbe):** nur die linke Hälfte bis zum schwarzen Balken. Rechts unten ist nichts Relevantes zu sehen.
- **SW (Schwarz-Weiß):** das Fenster, links/oben/rechts 10 px eingerückt, damit keine Randstörungen hineinkommen.
- **Alles außerhalb der beiden Ansichten wird im fertigen Video schwarz** (im Kontrollvideo nur abgedunkelt).
- Beide Ansichten sind ~960 px breit und werden von SAM 3 kaum verkleinert. Köpfe erscheinen so etwa doppelt so groß wie auf dem ganzen 1920er Bild.

### Ausgelaufene Masken

Manchmal wächst die Maske eines verfolgten Kopfes über Kissen, Decke oder Körper (Box breiter/höher als `MAX_FRAC` der Ansicht). Dann:

| Fall | Was wird verpixelt | Farbe im Kontrollvideo |
|---|---|---|
| normale Kopfmaske | die Maske | Farbe der ID |
| zu groß, **gute Kopfbox derselben ID** aus den letzten `MAX_AGE` Frames vorhanden | Maske, zugeschnitten auf die um `EXPAND` erweiterte letzte gute Box („clip“) | **orange** |
| zu groß, **keine gute Kopfbox** | die ganze große Maske („full“), lieber zu viel als zu wenig | **magenta** |

Der Umriss der ursprünglichen großen Maske wird zusätzlich dünn **rot** eingezeichnet.

In [ ]:
# Bildaufbau VID_76.AVI (Werte in Pixeln: x0, y0, x1, y1)
SPLIT_X = 961                    # ab hier beginnt rechts der schwarze Balken + S/W-Fenster
SW_VIEW = (991, 10, 1910, 527)   # S/W-Bild, links/oben/rechts 10 px eingerückt (unten passt)
VIEWS = {
    "F":  {"box": (0, 0, SPLIT_X, 1080), "blank": None},   # Farbe: nur linke Hälfte
    "SW": {"box": SW_VIEW,              "blank": None},   # "blank": optionales Rechteck, das vor SAM 3 geschwärzt wird
}
MAX_FRAC = 0.6   # Box breiter/höher als 60 % der Ansicht -> Maske ist "ausgelaufen"
EXPAND   = 0.5   # letzte gute Kopfbox an jeder Seite um 50 % ihrer Größe erweitern (-> doppelt so groß)
MAX_AGE  = 50    # letzte gute Kopfbox höchstens so viele Frames alt (50 = 2 s)

# Farben im Kontrollvideo (BGR)
COL_CLIP = (0, 165, 255)   # orange: ausgelaufene Maske, auf letzte gute Kopfbox zugeschnitten
COL_FULL = (255, 0, 255)   # magenta: ausgelaufene Maske ohne gute Kopfbox -> ganz verpixelt
COL_RAW  = (0, 0, 255)     # rot: Umriss der ursprünglichen, zu großen Maske


def put_label(img, text, org, color, scale=0.9):
    """Beschriftung mit dunklem Rand, damit sie auch auf eingefärbten Flächen lesbar ist."""
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), 5, cv2.LINE_AA)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2, cv2.LINE_AA)


def analysed_area(h, w):
    """True, wo eine Ansicht liegt. Alles andere wird im fertigen Video schwarz."""
    area = np.zeros((h, w), dtype=bool)
    for cfg in VIEWS.values():
        x0, y0, x1, y1 = cfg["box"]
        area[y0:y1, x0:x1] = True
    return area


def new_session(prompt):
    """Eine neue, leere SAM-3-Sitzung mit Text-Prompt."""
    s = video_processor.init_video_session(
        inference_device=DEVICE, video_storage_device="cpu", dtype=torch.bfloat16
    )
    return video_processor.add_text_prompt(s, prompt)


def clip_to_last_good(m, last_box, vw, vh, expand=EXPAND):
    """Schneidet eine ausgelaufene Maske auf die erweiterte letzte gute Kopfbox zu.
    Bleibt dabei zu wenig übrig (Kopf hat sich bewegt), wird die ganze erweiterte Box genommen."""
    lx1, ly1, lx2, ly2 = last_box
    pw, ph = (lx2 - lx1) * expand, (ly2 - ly1) * expand
    rx1, ry1 = int(max(0, lx1 - pw)), int(max(0, ly1 - ph))
    rx2, ry2 = int(min(vw, lx2 + pw)), int(min(vh, ly2 + ph))
    rect = np.zeros_like(m)
    rect[ry1:ry2, rx1:rx2] = True
    clipped = m & rect
    if clipped.sum() < 0.3 * (lx2 - lx1) * (ly2 - ly1):
        clipped = rect
    return clipped, [rx1, ry1, rx2, ry2]


def run_video_split(prompt, start=0, end=None, block=16, max_frac=MAX_FRAC, max_age=MAX_AGE):
    """Farb- und S/W-Ansicht getrennt (je eigene Sitzung). Speichert auch bei Abbruch."""
    cap = cv2.VideoCapture(str(INPUT_VIDEO))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    end = total if end is None else min(end, total)
    for _ in range(start):          # bis zum Start vorlesen (bei AVI ist cap.set ungenau)
        cap.grab()

    sessions = {v: new_session(prompt) for v in VIEWS}   # eine Sitzung pro Ansicht
    outside = ~analysed_area(h, w)                       # wird nicht ausgewertet -> schwarz
    last_good = {}                                       # (Ansicht, ID) -> (Box in Ansichts-Koordinaten, Frame-Index)

    stamp = time.strftime("%Y-%m-%d_%H-%M-%S")   # Datum + Uhrzeit -> nichts wird überschrieben
    stem = f"{INPUT_VIDEO.stem}_sam3split_{prompt}_{start}_{end}_{stamp}"
    print("Dateien:", stem)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer_pix = cv2.VideoWriter(str(OUTPUT_DIR / f"{stem}_pixel.mp4"), fourcc, fps, (w, h))
    writer_vis = cv2.VideoWriter(str(OUTPUT_DIR / f"{stem}_masken.mp4"), fourcc, fps, (w, h))

    per_frame = []
    n = end - start
    t0 = time.time()
    try:
        for i in range(n):
            ok, frame_bgr = cap.read()
            if not ok:
                break
            frame_nr = start + i

            small = cv2.resize(frame_bgr, (w // block, h // block), interpolation=cv2.INTER_LINEAR)
            pix = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
            frame_pix = frame_bgr.copy()
            frame_pix[outside] = 0                                               # fertiges Video: schwarz
            frame_vis = frame_bgr.copy()
            frame_vis[outside] = (frame_vis[outside] * 0.25).astype(np.uint8)    # Kontrollvideo: abgedunkelt
            objs, rejected = [], []

            for v, cfg in VIEWS.items():
                x0, y0, x1, y1 = cfg["box"]
                vw, vh = x1 - x0, y1 - y0

                # 1. Ausschnitt holen; optional ein Rechteck darin schwärzen
                view = frame_bgr[y0:y1, x0:x1].copy()
                if cfg["blank"] is not None:
                    ix0, iy0, ix1, iy1 = cfg["blank"]
                    view[iy0 - y0:iy1 - y0, ix0 - x0:ix1 - x0] = 0

                # 2. Durchs Modell (eigene Sitzung pro Ansicht)
                inputs = video_processor(images=cv2.cvtColor(view, cv2.COLOR_BGR2RGB), return_tensors="pt")
                with torch.inference_mode():
                    out = video_model(inference_session=sessions[v], frame_idx=i, frame=inputs.pixel_values[0])
                r = video_processor.postprocess_outputs(sessions[v], out, original_sizes=inputs.original_sizes)
                prune_session(sessions[v], i)

                # 3. Jede Maske prüfen, ggf. zuschneiden, verpixeln
                masks = r["masks"].cpu().numpy()     # Koordinaten der Ansicht
                for m, oid, sc, box in zip(masks, r["object_ids"].tolist(),
                                           r["scores"].tolist(), r["boxes"].cpu().tolist()):
                    key = (v, oid)
                    big = (box[2] - box[0]) > max_frac * vw or (box[3] - box[1]) > max_frac * vh

                    if not big:
                        mode = "ok"                                  # normale Kopfmaske
                        last_good[key] = (box, i)
                    else:
                        raw = [int(box[0]) + x0, int(box[1]) + y0, int(box[2]) + x0, int(box[3]) + y0]
                        rejected.append({"view": v, "id": oid, "score": sc, "box": raw})
                        cv2.rectangle(frame_vis, (raw[0], raw[1]), (raw[2], raw[3]), COL_RAW, 1)

                        lg = last_good.get(key)
                        if lg is not None and i - lg[1] <= max_age:
                            m, box = clip_to_last_good(m, lg[0], vw, vh)   # auf Kopfbereich zuschneiden
                            mode = "clip"
                        else:
                            mode = "full"                                # keine gute Box -> alles verpixeln

                    g = [int(box[0]) + x0, int(box[1]) + y0, int(box[2]) + x0, int(box[3]) + y0]   # ins Gesamtbild
                    frame_pix[y0:y1, x0:x1][m] = pix[y0:y1, x0:x1][m]          # verpixeln

                    if mode == "ok":
                        color = [int(c) for c in np.random.default_rng(2 * oid + (v == "SW")).integers(64, 256, 3)]
                    else:
                        color = COL_CLIP if mode == "clip" else COL_FULL
                    vis_view = frame_vis[y0:y1, x0:x1]
                    vis_view[m] = (0.5 * vis_view[m] + 0.5 * np.array(color)).astype(np.uint8)
                    cv2.rectangle(frame_vis, (g[0], g[1]), (g[2], g[3]), color, 2)
                    label = f"{v}{oid} {sc:.2f}" if mode == "ok" else f"{v}{oid} {mode} {sc:.2f}"
                    put_label(frame_vis, label, (g[0], max(g[1] - 8, 26)), color)
                    objs.append({"view": v, "id": oid, "score": sc, "box": g, "mode": mode})

            # Grenzen der Ansichten einzeichnen (zur Kontrolle)
            for cfg in VIEWS.values():
                x0, y0, x1, y1 = cfg["box"]
                cv2.rectangle(frame_vis, (x0, y0), (x1 - 1, y1 - 1), (255, 255, 255), 1)
            put_label(frame_vis, f"Frame {frame_nr}", (20, 40), (255, 255, 255), scale=1.0)
            writer_pix.write(frame_pix)
            writer_vis.write(frame_vis)
            per_frame.append({"frame": frame_nr, "objs": objs, "rejected": rejected})

            if i % 50 == 0:
                el = time.time() - t0
                cnt = {v: sum(o["view"] == v for o in objs) for v in VIEWS}
                n_clip = sum(o["mode"] == "clip" for o in objs)
                n_full = sum(o["mode"] == "full" for o in objs)
                print(f"Frame {frame_nr:5d} ({i}/{n})  F {cnt['F']}  SW {cnt['SW']}  (clip {n_clip}, full {n_full})  "
                      f"{el:7.1f} s  GPU {torch.cuda.memory_allocated() / 1e9:.2f} / "
                      f"reserviert {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    finally:
        # Läuft immer, auch beim Stopp-Knopf -> Videos abspielbar, Ergebnisse gespeichert
        cap.release()
        writer_pix.release()
        writer_vis.release()
        el = time.time() - t0
        print(f"Ende: {len(per_frame)} Frames in {el:.1f} s ({len(per_frame) / max(el, 1e-9):.2f} Frames/s)")
        with open(INTERIM_DIR / f"{stem}.pkl", "wb") as f:
            pickle.dump(per_frame, f)
    return per_frame

In [ ]:
def _longest_run(flags):
    """Längste Folge von True hintereinander."""
    longest, cur = 0, 0
    for f in flags:
        cur = cur + 1 if f else 0
        longest = max(longest, cur)
    return longest


def gap_stats_split(per_frame, name):
    """Pro Ansicht:
    - ohne Verpixelung: in dieser Ansicht wurde gar nichts verpixelt (echte Lücke, Ziel 0)
    - ohne frische Kopfmaske: kein Kopf mit normaler Maske in diesem Frame ('ok'); dann greift eine Notlösung
    - Notlösungen: clip/full (Tracking) bzw. hold/full (Erkennung pro Frame), Anzahl Frames"""
    for v, label in [("F", "Farbe"), ("SW", "S/W")]:
        empty = [not any(o["view"] == v for o in f["objs"]) for f in per_frame]
        no_ok = [not any(o["view"] == v and o["mode"] == "ok" for o in f["objs"]) for f in per_frame]
        modes = {}
        for f in per_frame:
            for m in {o["mode"] for o in f["objs"] if o["view"] == v and o["mode"] != "ok"}:
                modes[m] = modes.get(m, 0) + 1
        ids = sorted({o["id"] for f in per_frame for o in f["objs"] if o["view"] == v and o["id"] >= 0})
        print(f"{name} {label:6s}: ohne Verpixelung {sum(empty):4d} (längste Lücke {_longest_run(empty)}) | "
              f"ohne frische Kopfmaske {sum(no_ok):4d} (längste {_longest_run(no_ok)}) | "
              f"Notlösungen {dict(sorted(modes.items()))} | IDs {ids}")


def size_report(per_frame):
    """Größen der normalen und der zu großen (ausgelaufenen) Masken als Anteil der Ansicht."""
    for v, label in [("F", "Farbe"), ("SW", "S/W")]:
        x0, y0, x1, y1 = VIEWS[v]["box"]
        vw, vh = x1 - x0, y1 - y0
        print(f"--- {label} (Ansicht {vw} x {vh})")
        groups = {
            "normal   ": [o for f in per_frame for o in f["objs"] if o["view"] == v and o["mode"] == "ok"],
            "zu groß  ": [o for f in per_frame for o in f["rejected"] if o["view"] == v],
        }
        for name, items in groups.items():
            if not items:
                print(f"  {name}: keine")
                continue
            bw = np.array([(o["box"][2] - o["box"][0]) / vw for o in items])
            bh = np.array([(o["box"][3] - o["box"][1]) / vh for o in items])
            sc = np.array([o["score"] for o in items])
            ids = {}
            for o in items:
                ids[o["id"]] = ids.get(o["id"], 0) + 1
            print(f"  {name}: {len(items):5d} Masken | Breite min/median/max {np.percentile(bw, [0, 50, 100]).round(2)}"
                  f" | Höhe {np.percentile(bh, [0, 50, 100]).round(2)} | Score {np.percentile(sc, [0, 50, 100]).round(2)}"
                  f" | IDs {dict(sorted(ids.items()))}")

## 1b. Erkennung pro Frame (ohne Tracking)

Die Stichprobe A3 hat gezeigt: In der S/W-Ansicht findet SAM 3 **pro Einzelbild** den Kopf in 60 von 60 Frames sauber (Score 0,82–0,92, nie zu groß). Mit **Tracking** (`Sam3VideoModel`) war die Maske nur in 32 von 60 Frames normal, sonst „ausgelaufen“. Das Tracking trägt eine einmal ausgelaufene Maske von Frame zu Frame weiter.

`run_video_detect` nutzt deshalb das **Einzelbild-Modell** (`Sam3Model`) in jedem Frame und jeder Ansicht und ersetzt das Tracking durch einfache Regeln:

| Regel | Einstellung |
|---|---|
| Erkennung zählt ab Score | `DET_THR` |
| Erkennung gehört zum selben Kopf wie im Frame davor, wenn die Boxen sich so stark überlappen | `MATCH_IOU` (gibt stabile IDs, wichtig fürs spätere Face Swapping) |
| Wird ein Kopf kurz nicht erkannt, wird seine letzte Maske so viele Frames weiter verpixelt | `HOLD` (**gelb** im Kontrollvideo) |
| Masken werden um so viele Pixel vergrößert (Haaransatz, Ohren) | `DILATE` |
| Zu große Masken (wie in Abschnitt 1) werden zur Sicherheit ganz verpixelt | `MAX_FRAC` (**magenta**) |

Braucht Abschnitt 0 (ohne Video-Modell ginge es auch) und 1 (für `VIEWS`, `MAX_FRAC`, Farben, `put_label`). Lädt das Einzelbild-Modell zusätzlich (~1,7 GB GPU).

In [ ]:
from transformers import Sam3Model, Sam3Processor

DET_THR   = 0.5    # Mindest-Score einer Erkennung
MATCH_IOU = 0.3    # Box-Überlappung, ab der eine Erkennung derselbe Kopf ist wie im Frame davor
HOLD      = 25     # so viele Frames wird die letzte Maske nach dem letzten Treffer weiter verpixelt (25 = 1 s)
DILATE    = 8      # Masken um so viele Pixel vergrößern (Haaransatz, Ohren)
COL_HOLD  = (0, 255, 255)   # gelb: Kopf in diesem Frame nicht erkannt, letzte Maske gehalten
MASK_DIR  = INTERIM_DIR / "eval" / "masks"   # verpixelte Fläche einzelner Frames für Evaluation.ipynb

det_processor = Sam3Processor.from_pretrained(MODEL_DIR)
det_model = Sam3Model.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to(DEVICE).eval()
DILATE_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * DILATE + 1, 2 * DILATE + 1))
print(f"Einzelbild-Modell geladen, GPU belegt: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def detect_heads(view_bgr, prompt, threshold=DET_THR):
    """Einzelbild-Erkennung ohne Tracking -> Masken (bool), Boxen, Scores in Ansichts-Koordinaten."""
    inputs = det_processor(images=cv2.cvtColor(view_bgr, cv2.COLOR_BGR2RGB), text=prompt,
                           return_tensors="pt").to(DEVICE, dtype=torch.bfloat16)
    with torch.no_grad():
        out = det_model(**inputs)
    r = det_processor.post_process_instance_segmentation(
        out, threshold=threshold, mask_threshold=0.5, target_sizes=inputs["original_sizes"].tolist()
    )[0]
    return r["masks"].cpu().numpy().astype(bool), r["boxes"].cpu().tolist(), r["scores"].float().cpu().tolist()


def box_iou(a, b):
    """Überlappung zweier Boxen (0 = keine, 1 = identisch)."""
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def run_video_detect(prompt, start=0, end=None, block=16, max_frac=MAX_FRAC, hold=HOLD, match_iou=MATCH_IOU,
                     save_masks=()):
    """Erkennung pro Frame und Ansicht (ohne SAM-3-Tracking), eigene ID-Zuordnung und Halten bei Aussetzern.

    save_masks: Frame-Nummern, deren verpixelte Fläche (True/False je Pixel) als .npz in MASK_DIR
                gespeichert wird (für Evaluation.ipynb). Leer = nichts speichern.
    """
    cap = cv2.VideoCapture(str(INPUT_VIDEO))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    end = total if end is None else min(end, total)
    for _ in range(start):          # bis zum Start vorlesen (bei AVI ist cap.set ungenau)
        cap.grab()

    outside = ~analysed_area(h, w)
    tracks = {v: [] for v in VIEWS}      # pro Ansicht: Liste bekannter Köpfe (id, box, mask, score, last)
    next_id = {v: 0 for v in VIEWS}

    stamp = time.strftime("%Y-%m-%d_%H-%M-%S")   # Datum + Uhrzeit -> nichts wird überschrieben
    stem = f"{INPUT_VIDEO.stem}_sam3det_{prompt}_{start}_{end}_{stamp}"
    print("Dateien:", stem)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer_pix = cv2.VideoWriter(str(OUTPUT_DIR / f"{stem}_pixel.mp4"), fourcc, fps, (w, h))
    writer_vis = cv2.VideoWriter(str(OUTPUT_DIR / f"{stem}_masken.mp4"), fourcc, fps, (w, h))

    per_frame = []
    n = end - start
    t0 = time.time()
    try:
        for i in range(n):
            ok, frame_bgr = cap.read()
            if not ok:
                break
            frame_nr = start + i

            small = cv2.resize(frame_bgr, (w // block, h // block), interpolation=cv2.INTER_LINEAR)
            pix = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
            frame_pix = frame_bgr.copy()
            frame_pix[outside] = 0
            pix_mask = np.zeros((h, w), dtype=bool)   # alles, was in diesem Frame verpixelt wird
            frame_vis = frame_bgr.copy()
            frame_vis[outside] = (frame_vis[outside] * 0.25).astype(np.uint8)
            objs, rejected = [], []

            for v, cfg in VIEWS.items():
                x0, y0, x1, y1 = cfg["box"]
                vw, vh = x1 - x0, y1 - y0

                def paint(m, mode, oid, sc, box):
                    """Maske verpixeln und im Kontrollvideo einzeichnen."""
                    frame_pix[y0:y1, x0:x1][m] = pix[y0:y1, x0:x1][m]
                    pix_mask[y0:y1, x0:x1] |= m
                    if mode == "ok":
                        color = [int(c) for c in np.random.default_rng(2 * oid + (v == "SW")).integers(64, 256, 3)]
                    else:
                        color = COL_HOLD if mode == "hold" else COL_FULL
                    vis_view = frame_vis[y0:y1, x0:x1]
                    vis_view[m] = (0.5 * vis_view[m] + 0.5 * np.array(color)).astype(np.uint8)
                    g = [int(box[0]) + x0, int(box[1]) + y0, int(box[2]) + x0, int(box[3]) + y0]
                    cv2.rectangle(frame_vis, (g[0], g[1]), (g[2], g[3]), color, 2)
                    label = f"{v}{oid} {sc:.2f}" if mode == "ok" else f"{v}{oid} {mode} {sc:.2f}"
                    put_label(frame_vis, label, (g[0], max(g[1] - 8, 26)), color)
                    objs.append({"view": v, "id": oid, "score": sc, "box": g, "mode": mode})

                # 1. Ausschnitt holen und erkennen (jeder Frame neu, kein Gedächtnis)
                view = frame_bgr[y0:y1, x0:x1].copy()
                if cfg["blank"] is not None:
                    ix0, iy0, ix1, iy1 = cfg["blank"]
                    view[iy0 - y0:iy1 - y0, ix0 - x0:ix1 - x0] = 0
                masks, boxes, scores = detect_heads(view, prompt)

                # 2. Zu große Masken zur Sicherheit ganz verpixeln, normale weiterverarbeiten
                normal = []
                for m, b, s in zip(masks, boxes, scores):
                    if (b[2] - b[0]) > max_frac * vw or (b[3] - b[1]) > max_frac * vh:
                        raw = [int(b[0]) + x0, int(b[1]) + y0, int(b[2]) + x0, int(b[3]) + y0]
                        rejected.append({"view": v, "id": -1, "score": s, "box": raw})
                        paint(m, "full", -1, s, b)
                    else:
                        normal.append((cv2.dilate(m.astype(np.uint8), DILATE_KERNEL).astype(bool), b, s))

                # 3. Erkennungen den bekannten Köpfen zuordnen (größte Box-Überlappung zuerst)
                pairs = sorted(((box_iou(t["box"], d[1]), ti, di)
                                for ti, t in enumerate(tracks[v]) for di, d in enumerate(normal)), reverse=True)
                used_t, used_d = set(), set()
                for iou, ti, di in pairs:
                    if iou < match_iou:
                        break
                    if ti in used_t or di in used_d:
                        continue
                    used_t.add(ti)
                    used_d.add(di)
                    m, b, s = normal[di]
                    tracks[v][ti].update(mask=m, box=b, score=s, last=i)
                for di, (m, b, s) in enumerate(normal):
                    if di not in used_d:                     # neuer Kopf
                        tracks[v].append({"id": next_id[v], "mask": m, "box": b, "score": s, "last": i})
                        next_id[v] += 1

                # 4. Köpfe verpixeln: frisch erkannt ("ok") oder kurz gehalten ("hold"); zu alte vergessen
                tracks[v] = [t for t in tracks[v] if i - t["last"] <= hold]
                for t in tracks[v]:
                    paint(t["mask"], "ok" if t["last"] == i else "hold", t["id"], t["score"], t["box"])

            for cfg in VIEWS.values():
                x0, y0, x1, y1 = cfg["box"]
                cv2.rectangle(frame_vis, (x0, y0), (x1 - 1, y1 - 1), (255, 255, 255), 1)
            put_label(frame_vis, f"Frame {frame_nr}", (20, 40), (255, 255, 255), scale=1.0)
            writer_pix.write(frame_pix)
            writer_vis.write(frame_vis)
            per_frame.append({"frame": frame_nr, "objs": objs, "rejected": rejected})
            if frame_nr in save_masks:   # für Evaluation.ipynb: genau die Fläche, die verpixelt wurde
                MASK_DIR.mkdir(parents=True, exist_ok=True)
                np.savez_compressed(MASK_DIR / f"{INPUT_VIDEO.stem}_f{frame_nr:05d}_sam3det.npz", mask=pix_mask,
                                    run=stem, det_thr=DET_THR, dilate=DILATE, hold=hold, max_frac=max_frac)
                print(f"Maske für Frame {frame_nr} gespeichert ({int(pix_mask.sum())} px verpixelt)")

            if i % 50 == 0:
                el = time.time() - t0
                cnt = {v: sum(o["view"] == v for o in objs) for v in VIEWS}
                n_hold = sum(o["mode"] == "hold" for o in objs)
                n_full = sum(o["mode"] == "full" for o in objs)
                print(f"Frame {frame_nr:5d} ({i}/{n})  F {cnt['F']}  SW {cnt['SW']}  (hold {n_hold}, full {n_full})  "
                      f"{el:7.1f} s  GPU {torch.cuda.memory_allocated() / 1e9:.2f} / "
                      f"reserviert {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    finally:
        cap.release()
        writer_pix.release()
        writer_vis.release()
        el = time.time() - t0
        print(f"Ende: {len(per_frame)} Frames in {el:.1f} s ({len(per_frame) / max(el, 1e-9):.2f} Frames/s)")
        with open(INTERIM_DIR / f"{stem}.pkl", "wb") as f:
            pickle.dump(per_frame, f)
    return per_frame

## 2. Läufe

Ergebnisse landen in `data/processed/PAT_1/` (`..._pixel.mp4`, `..._masken.mp4`) und `data/interim/PAT_1/` (`.pkl`).
Im Kontrollvideo (`_masken`) prüfen: Sitzt der **weiße Rahmen** auf dem S/W-Fenster? Was steckt hinter den **roten X-Boxen**?

In [ ]:
# Tracking (Abschnitt 1) auf der schwierigsten Stelle, ca. 11 min
res_split = run_video_split("head", start=2400, end=3000)
gap_stats_split(res_split, "Tracking")
size_report(res_split)

In [ ]:
# Erkennung pro Frame (Abschnitt 1b) auf derselben Stelle, erwartet ca. 7 min
res_det = run_video_detect("head", start=2400, end=3000)
gap_stats_split(res_det, "pro Frame")
size_report(res_det)

In [ ]:
# Vergleich aus den gespeicherten Ergebnissen (jeweils neuester Lauf auf 2400–3000), ohne neu zu rechnen
for kind, name in [("sam3split", "Tracking"), ("sam3det", "pro Frame")]:
    runs = sorted(INTERIM_DIR.glob(f"{INPUT_VIDEO.stem}_{kind}_head_2400_3000_*.pkl"))
    if not runs:
        print(f"{name}: noch kein Lauf")
        continue
    print(f"=== {name}: {runs[-1].name}")
    gap_stats_split(pickle.load(open(runs[-1], "rb")), name)

In [ ]:
# Ganzes Video mit Erkennung pro Frame: 8759 Frames, bei ~1,29 Frames/s ca. 1 h 55 min
# (bricht man ab, werden Videos und .pkl trotzdem sauber gespeichert)
res_full = run_video_detect("head")
gap_stats_split(res_full, "ganzes Video")
size_report(res_full)

### Masken für die Evaluation (markierte Frames)

Für jeden Frame, den du in X-AnyLabeling markiert hast (`data/interim/PAT_1/gt/*.json`), läuft die **komplette Pipeline** (`run_video_detect` mit Ansichtstrennung, hold, Vergrößerung) ab `LEAD` Frames **vor** diesem Frame. So ist beim markierten Frame derselbe Zustand erreicht wie im ganzen Lauf (gehaltene Masken reichen höchstens `HOLD` = 25 Frames zurück).

Gespeichert wird die verpixelte Fläche als `data/interim/PAT_1/eval/masks/VID_76_f<Frame>_sam3det.npz`. Die liest `Evaluation.ipynb` ein. Nebenbei entstehen kurze Videos und eine `.pkl` wie bei jedem Lauf.

Dauer: ca. 1 min pro markiertem Frame (51 Frames bei ~1,3 Frames/s, plus Vorspulen).

In [ ]:
GT_DIR = INTERIM_DIR / "gt"
GT_FRAMES = sorted(int(p.stem.split("_f")[-1]) for p in GT_DIR.glob(f"{INPUT_VIDEO.stem}_f*.json"))
LEAD = 2 * HOLD   # so viele Frames vorher starten (50 = 2 s)
print("Markierte Frames:", GT_FRAMES)

for f in GT_FRAMES:
    run_video_detect("head", start=max(0, f - LEAD), end=f + 1, save_masks={f})

## 3. Notizen

Ausführliches Log mit Umgebung, allen Versuchen und Entscheidungen: [`docs/SAM3_LOG.md`](../docs/SAM3_LOG.md)

**Kurzfassung (Frames 2400–3000):**

| Lauf | Verfahren | Farbe ohne frische Kopfmaske | S/W ohne frische Kopfmaske | S/W ausgelaufene Masken | Lücken | Tempo |
|---|---|---|---|---|---|---|
| v4 (`16-08-36`) | Tracking, getrennte Ansichten, clip/full, NMS-Ersatz | 2 | 293 (längste 27) | 952 | 0 / 0 | 0,86 Frames/s |
| **v5 (`16-53-21`)** | **pro Frame ohne Tracking** (`run_video_detect`) | **0** | **0** | **0** | 0 / 0 | **1,29 Frames/s** |

- v5: Patientenkopf in S/W (ID 0) in allen 600 Frames frisch erkannt. „hold“ nur bei kurzen Nebenerkennungen (z. B. Pflegekraft, schnell bewegt, kleiner Kopfteil sichtbar), visuell unproblematisch.
- Der `kernels`-Ersatz (Abschnitt 0) wirkt nur beim Tracking-Modell. Das Einzelbild-Modell braucht ihn nicht.

**Entscheidung:** `run_video_detect` ist das Standardverfahren.

**Nächster Schritt:** ganzes VID_76 mit `run_video_detect` (Abschnitt 2, letzte Zelle).

---
## Anhang: Vortests (nicht im normalen Ablauf ausführen)

Zum Nachlesen und Wiederholen. Die Einzelbild-Tests laden ein **zweites Modell** (`Sam3Model`, ~1,7 GB GPU).

### A1. Einzelbild-Test: „face“ vs. „head“

In [ ]:
from PIL import Image
from transformers import Sam3Model, Sam3Processor

processor = Sam3Processor.from_pretrained(MODEL_DIR)
model = Sam3Model.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to(DEVICE).eval()

FRAME_NR = 250
cap = cv2.VideoCapture(str(INPUT_VIDEO))
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_NR)
ok, frame_bgr = cap.read()
cap.release()
assert ok, "Frame konnte nicht gelesen werden"
image = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))   # OpenCV: BGR, SAM 3: RGB


def detect(image, prompt, threshold=0.3):
    """Sucht alle Objekte, die zum Text-Prompt passen. Gibt Masken, Boxen und Scores zurück."""
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE, dtype=torch.bfloat16)
    with torch.no_grad():
        outputs = model(**inputs)
    return processor.post_process_instance_segmentation(
        outputs, threshold=threshold, mask_threshold=0.5,
        target_sizes=inputs["original_sizes"].tolist(),
    )[0]


results = {}
for prompt in ["face", "head"]:
    r = detect(image, prompt)
    results[prompt] = r
    print(f"{prompt!r}: {len(r['scores'])} Treffer")
    for box, score in zip(r["boxes"].tolist(), r["scores"].tolist()):
        x1, y1, x2, y2 = map(int, box)
        print(f"    Score {score:.2f}   Box ({x1},{y1})-({x2},{y2})   Größe {x2 - x1} x {y2 - y1}")

# Ergebnisbilder speichern (nicht im Notebook anzeigen: Patientendaten)
colors = {"face": (0, 255, 0), "head": (0, 165, 255)}
for prompt, r in results.items():
    vis, out = frame_bgr.copy(), frame_bgr.copy()
    h, w = out.shape[:2]
    pix = cv2.resize(cv2.resize(out, (w // 16, h // 16)), (w, h), interpolation=cv2.INTER_NEAREST)
    for mask, box, score in zip(r["masks"], r["boxes"], r["scores"]):
        m = mask.cpu().numpy().astype(bool)
        vis[m] = (0.5 * vis[m] + 0.5 * np.array(colors[prompt])).astype(np.uint8)
        out[m] = pix[m]
    cv2.imwrite(str(OUTPUT_DIR / f"sam3_frame{FRAME_NR}_{prompt}_masken.png"), vis)
    cv2.imwrite(str(OUTPUT_DIR / f"sam3_frame{FRAME_NR}_{prompt}_pixel.png"), out)

del model                       # GPU-Speicher wieder freigeben
torch.cuda.empty_cache()

### A2. Tracking auf dem ganzen Bild (ohne getrennte Ansichten)

Erster Video-Ansatz. Funktioniert, verfolgt aber auf VID_76 viele Fehlalarme (siehe Notizen). Ersetzt durch `run_video_split`.

In [ ]:
def run_video(prompt, start=0, end=None, block=16):
    """SAM 3 mit Tracking über Frames [start, end) auf dem ganzen Bild."""
    cap = cv2.VideoCapture(str(INPUT_VIDEO))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    end = total if end is None else min(end, total)
    for _ in range(start):
        cap.grab()

    stamp = time.strftime("%Y-%m-%d_%H-%M-%S")   # Datum + Uhrzeit -> nichts wird überschrieben
    stem = f"{INPUT_VIDEO.stem}_sam3_{prompt}_{start}_{end}_{stamp}"
    print("Dateien:", stem)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer_pix = cv2.VideoWriter(str(OUTPUT_DIR / f"{stem}_pixel.mp4"), fourcc, fps, (w, h))
    writer_vis = cv2.VideoWriter(str(OUTPUT_DIR / f"{stem}_masken.mp4"), fourcc, fps, (w, h))
    session = new_session(prompt)

    per_frame = []
    n = end - start
    t0 = time.time()
    try:
        for i in range(n):
            ok, frame_bgr = cap.read()
            if not ok:
                break
            frame_nr = start + i
            inputs = video_processor(images=cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB), return_tensors="pt")
            with torch.inference_mode():
                out = video_model(inference_session=session, frame_idx=i, frame=inputs.pixel_values[0])
            r = video_processor.postprocess_outputs(session, out, original_sizes=inputs.original_sizes)
            prune_session(session, i)

            ids, scores, boxes = r["object_ids"].tolist(), r["scores"].tolist(), r["boxes"].cpu().tolist()
            per_frame.append({"frame": frame_nr, "ids": ids, "scores": scores, "boxes": boxes})

            masks = r["masks"].cpu().numpy()
            pix = cv2.resize(cv2.resize(frame_bgr, (w // block, h // block)), (w, h), interpolation=cv2.INTER_NEAREST)
            frame_pix, frame_vis = frame_bgr.copy(), frame_bgr.copy()
            for m, oid, sc, box in zip(masks, ids, scores, boxes):
                frame_pix[m] = pix[m]
                color = [int(c) for c in np.random.default_rng(oid).integers(64, 256, 3)]
                frame_vis[m] = (0.5 * frame_vis[m] + 0.5 * np.array(color)).astype(np.uint8)
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(frame_vis, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame_vis, f"ID {oid}  {sc:.2f}", (x1, max(y1 - 8, 20)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            cv2.putText(frame_vis, f"Frame {frame_nr}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
            writer_pix.write(frame_pix)
            writer_vis.write(frame_vis)

            if i % 50 == 0:
                print(f"Frame {frame_nr:5d} ({i}/{n})  {len(ids)} Objekt(e)  {time.time() - t0:7.1f} s  "
                      f"GPU {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    finally:
        cap.release()
        writer_pix.release()
        writer_vis.release()
        print(f"Ende: {len(per_frame)} Frames in {time.time() - t0:.1f} s")
        with open(INTERIM_DIR / f"{stem}.pkl", "wb") as f:
            pickle.dump(per_frame, f)
    return per_frame


def gap_stats(per_frame, name):
    """Frames ohne Treffer, längster Aussetzer und alle vorkommenden Objekt-IDs."""
    empty = [len(f["ids"]) == 0 for f in per_frame]
    longest, cur = 0, 0
    for e in empty:
        cur = cur + 1 if e else 0
        longest = max(longest, cur)
    all_ids = sorted({i for f in per_frame for i in f["ids"]})
    print(f"{name:6s}: {sum(empty):4d} von {len(per_frame)} Frames ohne Treffer "
          f"({100 * sum(empty) / len(per_frame):.1f} %), längster Aussetzer {longest} Frames, IDs: {all_ids}")

# Beispiel: res_head = run_video("head", start=8000, end=8300); gap_stats(res_head, "head")

### A3. S/W-Ansicht ohne Tracking (Stichprobe)

Frage: Läuft die S/W-Kopfmaske aus, weil das **Tracking** sie falsch weiterträgt, oder findet SAM 3 auch **pro Einzelbild** keine saubere Kopfmaske?

Jeder 10. Frame von 2400–3000 (60 Frames), nur die S/W-Ansicht, mit dem **Einzelbild-Modell** (`Sam3Model`, kein Gedächtnis über Frames). Gezählt wird pro Frame: Köpfe normaler Größe, zu große Masken (gleiche Grenze `MAX_FRAC` wie in Abschnitt 1). Daneben steht, was das Tracking (letzter `run_video_split`-Lauf auf 2400–3000) im selben Frame in der S/W-Ansicht geliefert hat.

Braucht Abschnitt 0 und 1 (für `SW_VIEW`, `MAX_FRAC`). Lädt ein zweites Modell (~1,7 GB GPU) und gibt es am Ende wieder frei. Dauer ca. 1 min.

In [ ]:
from transformers import Sam3Model, Sam3Processor

img_processor = Sam3Processor.from_pretrained(MODEL_DIR)
img_model = Sam3Model.from_pretrained(MODEL_DIR, dtype=torch.bfloat16).to(DEVICE).eval()


def detect_view(view_bgr, prompt="head", threshold=0.3):
    """Einzelbild-Erkennung ohne Tracking. Gibt Masken, Boxen und Scores zurück."""
    inputs = img_processor(images=cv2.cvtColor(view_bgr, cv2.COLOR_BGR2RGB), text=prompt,
                           return_tensors="pt").to(DEVICE, dtype=torch.bfloat16)
    with torch.no_grad():
        out = img_model(**inputs)
    return img_processor.post_process_instance_segmentation(
        out, threshold=threshold, mask_threshold=0.5, target_sizes=inputs["original_sizes"].tolist()
    )[0]


START, END, STEP = 2400, 3000, 10
x0, y0, x1, y1 = SW_VIEW
vw, vh = x1 - x0, y1 - y0

# Zum Vergleich: was das Tracking im letzten Lauf auf denselben Frames in der S/W-Ansicht geliefert hat
pkl = sorted(INTERIM_DIR.glob(f"{INPUT_VIDEO.stem}_sam3split_head_{START}_{END}_*.pkl"))[-1]
track = {f["frame"]: f for f in pickle.load(open(pkl, "rb"))}
print("Vergleich mit Tracking-Lauf:", pkl.name)

cap = cv2.VideoCapture(str(INPUT_VIDEO))
for _ in range(START):
    cap.grab()
rows = []
t0 = time.time()
for nr in range(START, END):
    if (nr - START) % STEP:
        cap.grab()                    # Frame überspringen (schnell, ohne Dekodieren)
        continue
    ok, frame = cap.read()
    if not ok:
        break
    r = detect_view(frame[y0:y1, x0:x1])
    dets = [((b[2] - b[0]) / vw, (b[3] - b[1]) / vh, s) for b, s in zip(r["boxes"].tolist(), r["scores"].tolist())]
    normal = [d for d in dets if d[0] <= MAX_FRAC and d[1] <= MAX_FRAC]
    big = [d for d in dets if not (d[0] <= MAX_FRAC and d[1] <= MAX_FRAC)]
    modes = sorted(o["mode"] for o in track.get(nr, {"objs": []})["objs"] if o["view"] == "SW")
    rows.append((nr, len(normal), len(big), max((d[2] for d in normal), default=0.0), modes))
cap.release()
print(f"{len(rows)} Frames in {time.time() - t0:.1f} s\n")

print("Frame | normal | zu groß | bester Score (normal) | Tracking (S/W)")
for nr, n_ok, n_big, best, modes in rows:
    print(f"{nr:5d} | {n_ok:6d} | {n_big:7d} | {best:21.2f} | {modes}")

n = len(rows)
print(f"\nOhne Tracking: mind. 1 normaler Kopf in {sum(r[1] > 0 for r in rows)} von {n} Frames, "
      f"davon mit Score >= 0.5: {sum(r[3] >= 0.5 for r in rows)}; "
      f"zu große Maske in {sum(r[2] > 0 for r in rows)}; gar nichts in {sum(r[1] + r[2] == 0 for r in rows)}")
print(f"Mit Tracking:  normale S/W-Maske in {sum('ok' in r[4] for r in rows)} von {n} Frames")

del img_model                   # GPU-Speicher wieder freigeben
torch.cuda.empty_cache()